# 1. Inladen packages en models

## 1.1 Pip en Packages

In [ ]:
#required for smol
# !pip install num2words

In [ ]:
# !pip install transformers==4.57.1

In [ ]:
import requests
from PIL import Image
import torch
from transformers import AutoModelForImageTextToText, Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor, LlavaForConditionalGeneration
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
import string
from google.colab import files

from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


# 2. Import Dataset

In [ ]:
data = "insert link to csv here"
df = pd.read_csv(data, sep=",")


(383, 6)
0      100
1      110
2      120
3      160
4      170
      ... 
378    210
379    300
380    180
381    240
382    260
Name: Weight, Length: 383, dtype: int64
Index                                                         0
Site_link          https://height-weight-chart.com/410-090.html
Image_link    https://height-weight-chart.com/l/410-090_Del_...
Filename                                     410-090_Del_L1.jpg
Height                                                    4' 10
Weight                                                      100
Name: 0, dtype: object


# 3. Opzet Experiment

#smol part

In [ ]:
drive_model_path = "insert link to saved model here"

model = AutoModelForImageTextToText.from_pretrained(drive_model_path).to('cuda')
proc = AutoProcessor.from_pretrained(drive_model_path)

print("Model and processor loaded from Google Drive.")

results = pd.DataFrame(columns=[
    'index',
    'guess',
    'model',
    'temp',
    'top_p',
    'top_k',
    'true_weight',
    'diff'])
results = results.astype({
    'index': int,
    'guess': object,
    'model': str,
    'temp': float,
    'top_p': float,
    'top_k': float,
    'true_weight': str,
    'diff': float})
print(results.dtypes)

# Generate a fixed set of random indices
np.random.seed(23836187)
rd_sample_indices = np.random.randint(low=0, high=383, size=10, dtype=int)
print(rd_sample_indices)

index = 0
base_prompt = "Guess the weight of the person in the image, only output the weight in lbs"

temps = [0.6, 0.8, 1]
top_ps = [0.7, 1]
top_ks = [30, 70]

for t in temps:
    for tp in top_ps:
        for tk in top_ks:
            for sample_idx in rd_sample_indices:
                for j in range(25):

                    # generate random strings
                    random_letters = ''.join(random.choices(string.ascii_letters, k=10))

                    # compose new text, adding the random letters to the original prompt
                    modified_text = f"{random_letters} {base_prompt}"

                    # define full prompt and system message
                    current_messages = [
                        {
                            "role": "system",
                            "content": "Based on an image, you guess the weight in lbs this person most likely weighs"
                        },
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": modified_text},
                                {"type": "image"}
                            ]
                        }
                    ]

                    # apply model chat template to prompt
                    text_prompt = proc.apply_chat_template(
                        current_messages,
                        add_generation_prompt=True,
                        tokenize=False
                    )

                    url = df.iloc[sample_idx]['Image_link']
                    raw_image = Image.open(requests.get(url, stream=True).raw)

                    inputs = proc(
                        text=text_prompt,
                        images=raw_image,
                        return_tensors='pt'
                    ).to(0, torch.float16)

                    output = model.generate(**inputs, max_new_tokens=13, do_sample=True, temperature=t, top_k=tk, top_p=tp)
                    output_decoded = proc.decode(output[0][0:], skip_special_tokens=True)

                    #remove "lbs" from output if possible
                    try:
                      guessed_weight = int(output_decoded[-3:])
                    except:
                      #print(output_decoded)
                      try:
                        guessed_weight = int(output_decoded[-1:])
                      except:
                        guessed_weight = output_decoded[-8:]

                    true_weight_value = df.iloc[sample_idx]['Weight']

                    results.loc[index] = {
                        'index': index,
                        'guess': guessed_weight,
                        'model': 'smol',
                        'temp': t,
                        'top_p': tp,
                        'top_k': tk,
                        'true_weight': true_weight_value,
                        'diff': 0,
                        'noise': random_letters  # could be useful for analysis
                    }

                    #try to print the difference between the guess and true_weight for debugging purposes, otherwise skip it
                    try:
                        diff = abs(guessed_weight - true_weight_value) if isinstance(guessed_weight, int) else "N/A"
                        print(index, t, f"Noise: {random_letters}", guessed_weight, true_weight_value, diff)
                    except:
                        print(index, guessed_weight)

                    index += 1

results.to_csv('EXP3-RAW_results_smol.csv', index=False)
files.download('EXP3-RAW_results_smol.csv')

Streaming output truncated to the last 5000 lines.
Assistant: The weight of the person is 178.2 lbs
2169 1 Noise: jqQSBXeLJS 78.2 lbs 140 N/A
System: 
User: JkTbVfkLlU Guess the weight of the person in the image, only output the weight in lbs





Assistant: 155.8
2170 1 Noise: JkTbVfkLlU 8 140 132
2171 1 Noise: xytxKCwEPK 130 140 10
2172 1 Noise: nLEnSeGVAb 160 140 20
System: 
User: ptvhpvcPAG Guess the weight of the person in the image, only output the weight in lbs





Assistant: 125.5
2173 1 Noise: ptvhpvcPAG 5 140 135
2174 1 Noise: rJchMgmzTC 17 140 123
2175 1 Noise: vFswLTbwfJ 16 160 144
System: 
User: zzilzAvdbq Guess the weight of the person in the image, only output the weight in lbs





Assistant: The weight of the person is 130 lbs.
2176 1 Noise: zzilzAvdbq 130 lbs. 160 N/A
System: 
User: FSTntISACi Guess the weight of the person in the image, only output the weight in lbs





Assistant: 160 lbs
2177 1 Noise: FSTntISACi  160 lbs 160 N/A
System: 
User: ieGmyziPXq Guess the

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# might be needed for qwen
# !pip install transformers==4.57.1

# Qwen part

In [ ]:
drive_model_path = ""

model = Qwen2VLForConditionalGeneration.from_pretrained(drive_model_path).to('cuda')
proc = AutoProcessor.from_pretrained(drive_model_path)

print("Model and processor loaded from Google Drive.")

results = pd.DataFrame(columns=[
    'index',
    'guess',
    'model',
    'temp',
    'top_p',
    'top_k',
    'true_weight',
    'diff'])
results = results.astype({
    'index': int,
    'guess': object,
    'model': str,
    'temp': float,
    'top_p': float,
    'top_k': float,
    'true_weight': str,
    'diff': float})
print(results.dtypes)

np.random.seed(23836187)
rd_sample_indices = np.random.randint(low=0, high=383, size=10, dtype=int)
print(rd_sample_indices)

index = 0
base_prompt = "Guess the weight of the person in the image, only output the weight in lbs"

temps = [0.6, 0.8, 1]
top_ps = [0.7, 1]
top_ks = [30, 70]

for t in temps:
    for tp in top_ps:
        for tk in top_ks:
            for sample_idx in rd_sample_indices:
                for j in range(25):

                    random_letters = ''.join(random.choices(string.ascii_letters, k=10))

                    modified_text = f"{random_letters} {base_prompt}"

                    current_messages = [
                        {
                            "role": "system",
                            "content": "Based on an image, you guess the weight in lbs this person most likely weighs"
                        },
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": modified_text},
                                {"type": "image"}
                            ]
                        }
                    ]

                    text_prompt = proc.apply_chat_template(
                        current_messages,
                        add_generation_prompt=True,
                        tokenize=False
                    )

                    url = df.iloc[sample_idx]['Image_link']
                    raw_image = Image.open(requests.get(url, stream=True).raw)

                    inputs = proc(
                        text=text_prompt,
                        images=raw_image,
                        return_tensors='pt'
                    ).to(0, torch.float16)

                    output = model.generate(**inputs, max_new_tokens=13, do_sample=True, temperature=t, top_k=tk, top_p=tp)
                    output_decoded = proc.decode(output[0][0:], skip_special_tokens=True)

                    try:
                      guessed_weight = int(output_decoded[-3:])
                    except:
                      try:
                        guessed_weight = int(output_decoded[-1:])
                      except:
                        guessed_weight = output_decoded[-8:]

                    true_weight_value = df.iloc[sample_idx]['Weight']

                    results.loc[index] = {
                        'index': index,
                        'guess': guessed_weight,
                        'model': 'qwen',
                        'temp': t,
                        'top_p': tp,
                        'top_k': tk,
                        'true_weight': true_weight_value,
                        'diff': 0,
                        'noise': random_letters
                    }

                    try:
                        diff = abs(guessed_weight - true_weight_value) if isinstance(guessed_weight, int) else "N/A"
                        print(index, t, f"Noise: {random_letters}", guessed_weight, true_weight_value, diff)
                    except:
                        print(index, guessed_weight)

                    index += 1

results.to_csv('EXP3-RAW_results_qwen.csv', index=False)
files.download('EXP3-RAW_results_qwen.csv')

Model and processor loaded from Google Drive.
index            int64
guess           object
model           object
temp           float64
top_p          float64
top_k          float64
true_weight     object
diff           float64
dtype: object
[ 87  38  34 276  93 128 279  23 310 310]
0 0.6 Noise: YAvUgbtIki 130 110 20
1 0.6 Noise: yvRqCFSJzP 145 110 35
2 0.6 Noise: ErzOqmJrfk 140 110 30
3 0.6 Noise: PdVjezZBFr 120 110 10
4 0.6 Noise: QSHdKkKyvT 130 110 20
5 0.6 Noise: nYHvchHmOy 140 110 30
6 0.6 Noise: FbjTTzILfO 125 110 15
7 0.6 Noise: VtTfkJFLQC 120 110 10
8 0.6 Noise: mJaWXVaJVU 140 110 30
9 0.6 Noise: zmUwqAmwEF 145 110 35
10 0.6 Noise: wIeWrPkNKJ 135 110 25
11 0.6 Noise: LQWDPmPbKr 120 110 10
12 0.6 Noise: EpardrQxFl 135 110 25
13 0.6 Noise: FSujGewMty 125 110 15
14 0.6 Noise: WLSGNMpkOH 130 110 20
15 0.6 Noise: ppTVAyReHd 145 110 35
16 0.6 Noise: kKpUyfsIgC 120 110 10
17 0.6 Noise: ebHZJHdCkQ 125 110 15
18 0.6 Noise: iFhKRlSYPC 125 110 15
19 0.6 Noise: tiPJJFpNef 140 110 30
20 0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# llava part

In [ ]:
# I was not able to get LLaVA saved locally so it is downloaded here instead of imported from drive
model_id3 = "llava-hf/llava-1.5-7b-hf"

model = LlavaForConditionalGeneration.from_pretrained(
   model_id3,
   dtype=torch.float16,
  low_cpu_mem_usage=True,
).to(0, torch.float16)

proc = AutoProcessor.from_pretrained(model_id3)

print("Model and processor downloaded")

results = pd.DataFrame(columns=[
    'index',
    'guess',
    'model',
    'temp',
    'top_p',
    'top_k',
    'true_weight',
    'diff'])
results = results.astype({
    'index': int,
    'guess': object,
    'model': str,
    'temp': float,
    'top_p': float,
    'top_k': float,
    'true_weight': str,
    'diff': float})

np.random.seed(23836187)
rd_sample_indices = np.random.randint(low=0, high=383, size=10, dtype=int)
print(rd_sample_indices)

index = 0
base_prompt = "Guess the weight of the person in the image, only output the weight in lbs"

temps = [0.6, 0.8, 1]
top_ps = [0.7, 1]
top_ks = [30, 70]

for t in temps:
    for tp in top_ps:
        for tk in top_ks:
            for sample_idx in rd_sample_indices:
                for j in range(25):

                    random_letters = ''.join(random.choices(string.ascii_letters, k=10))

                    modified_text = f"{random_letters} {base_prompt}"

                    current_messages = [
                        {
                            "role": "system",
                            "content": "Based on an image, you guess the weight in lbs this person most likely weighs"
                        },
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": modified_text},
                                {"type": "image"}
                            ]
                        }
                    ]

                    text_prompt = proc.apply_chat_template(
                        current_messages,
                        add_generation_prompt=True,
                        tokenize=False
                    )

                    url = df.iloc[sample_idx]['Image_link']
                    raw_image = Image.open(requests.get(url, stream=True).raw)

                    inputs = proc(
                        text=text_prompt,
                        images=raw_image,
                        return_tensors='pt'
                    ).to(0, torch.float16)

                    output = model.generate(**inputs, max_new_tokens=13, do_sample=True, temperature=t, top_k=tk, top_p=tp)
                    output_decoded = proc.decode(output[0][0:], skip_special_tokens=True)

                    try:
                      guessed_weight = int(output_decoded[-3:])
                    except:
                      try:
                        guessed_weight = int(output_decoded[-1:])
                      except:
                        guessed_weight = output_decoded[-8:]

                    true_weight_value = df.iloc[sample_idx]['Weight']

                    results.loc[index] = {
                        'index': index,
                        'guess': guessed_weight,
                        'model': 'llava',
                        'temp': t,
                        'top_p': tp,
                        'top_k': tk,
                        'true_weight': true_weight_value,
                        'diff': 0,
                        'noise': random_letters
                    }

                    try:
                        diff = abs(guessed_weight - true_weight_value) if isinstance(guessed_weight, int) else "N/A"
                        print(index, t, f"Noise: {random_letters}", guessed_weight, true_weight_value, diff)
                    except:
                        print(index, guessed_weight)

                    index += 1

results.to_csv('EXP3-RAW_results_llava.csv', index=False)
files.download('EXP3-RAW_results_llava.csv')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

Model and processor downloaded
index            int64
guess           object
model           object
temp           float64
top_p          float64
top_k          float64
true_weight     object
diff           float64
dtype: object
[ 87  38  34 276  93 128 279  23 310 310]
0 0.6 Noise: YVymBDxUBh 120 110 10
1 0.6 Noise: DjroGuewVB 120 110 10
2 0.6 Noise: CHNqYboFRl 130 110 20
3 0.6 Noise: NFpqtmBXLX 150 110 40
4 0.6 Noise: MSWiJMaRYI 150 110 40
5 0.6 Noise: MMKbLDwxPx 130 110 20
6 0.6 Noise: DchxFlOWos 120 110 10
7 0.6 Noise: EqvSeClgVl 150 110 40
8 0.6 Noise: UmROCowDyy  120 lbs 110 N/A
9 0.6 Noise: obRtEedNno 130 110 20
10 0.6 Noise: CEtszPGFll 150 110 40
11 0.6 Noise: jyvWbAQrkt 150 110 40
12 0.6 Noise: oerEhItMAA 150 110 40
13 0.6 Noise: hShKvIIdvD 150 110 40
14 0.6 Noise: ivUGtPJOKC 120 110 10
15 0.6 Noise: tcrzeuMyko 120 110 10
16 0.6 Noise: xMAuzMixPD 120 110 10
17 0.6 Noise: YNzizxmOVS 120 110 10
18 0.6 Noise: LbatEOIFGA 150 110 40
19 0.6 Noise: wxblGLHygF 150 110 40
20 0.6 Noise:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# from google.colab import runtime
# runtime.unassign()